# 01 · Fondations : parler à un LLM et obtenir du JSON

**Atelier Agents IA · Notebook 1 sur 4**

| | |
|---|---|
| ⏱️ Durée conseillée | 40 minutes |
| 🎓 Séquence du cours | Séquences 1 (Fondations) et 2 (Sorties structurées) |
| 🧰 Prérequis | Fichier `.env` rempli (voir le README) |

### Ce que vous saurez faire à la fin
- faire un premier appel à un modèle de langage depuis Python ;
- compter les tokens et estimer le coût d'un appel en FCFA ;
- comprendre la température et l'absence de mémoire du modèle ;
- obtenir une réponse JSON fiable et la valider avec Pydantic.

> **Comment utiliser ce notebook :** exécutez les cellules **une par une, dans l'ordre**, avec `Maj + Entrée`.
> Lisez le texte avant chaque cellule de code : il explique ce qui va se passer.
> Les cellules marquées ✍️ **À vous** sont des exercices ; la solution est cachée juste en dessous (cliquez sur « Voir la solution »).

## 0. Installation et configuration

### 0.1 Installer les bibliothèques
À faire **une seule fois** sur votre ordinateur. Cette cellule installe tout ce dont l'atelier a besoin (cela peut prendre 2 à 3 minutes).

In [20]:
#%pip install -q -r ../requirements.txt

### 0.2 Charger vos clés depuis le fichier `.env`
Vos clés API ne doivent **jamais** être écrites dans le code : elles sont rangées dans le fichier `.env` à la racine du dossier de l'atelier.
La fonction `load_dotenv` les lit et les place dans les variables d'environnement.

In [21]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # cherche le fichier .env dans ce dossier ou au-dessus

MODELE = os.getenv("MODELE", "claude-sonnet-5-5")
MODELE_RAPIDE = os.getenv("MODELE_RAPIDE") or MODELE

cle = os.getenv("ANTHROPIC_API_KEY", "")
print("Clé Anthropic :", "✅ trouvée" if cle else "❌ absente : ouvrez le fichier .env et collez votre clé")
print("Modèle utilisé :", MODELE)

Clé Anthropic : ✅ trouvée
Modèle utilisé : claude-sonnet-5-5


> ❗ **Si la clé est absente :** ouvrez le fichier `.env`, collez votre clé après `ANTHROPIC_API_KEY=`, enregistrez, puis **redémarrez le noyau** (menu *Kernel → Restart*) et relancez les cellules.

## 1. Premier appel au modèle

Un LLM (*Large Language Model*) reçoit du texte et **prédit la suite**. On lui envoie une liste de messages, il renvoie une réponse.

Le client `anthropic.Anthropic()` lit tout seul la clé `ANTHROPIC_API_KEY`.

In [22]:
import anthropic

client = anthropic.Anthropic()

reponse = client.messages.create(
    model=MODELE,
    max_tokens=300,                       # longueur maximale de la réponse
    messages=[{"role": "user", "content": "Explique en 2 phrases ce qu'est un agent IA."}],
)

# La réponse est une liste de blocs : le modèle peut d'abord « réfléchir » (bloc thinking)
# avant d'écrire son texte. On ne garde donc que les blocs de type "text".
def texte_de(rep):
    return "".join(bloc.text for bloc in rep.content if bloc.type == "text") #+ "\n\n" + str(rep.content)

print(texte_de(reponse))

Un agent IA est un système logiciel qui perçoit son environnement, prend des décisions et agit de façon autonome pour atteindre un objectif donné, souvent en utilisant des outils (recherche web, code, API). Contrairement à un simple chatbot qui répond à une question, il peut planifier une suite d'étapes, évaluer les résultats et ajuster son comportement jusqu'à accomplir sa tâche.


👀 **Regardez l'objet complet** : en plus du texte, la réponse contient `usage`, le nombre de tokens consommés. C'est ce qui est facturé.

In [23]:
print("Tokens en entrée :", reponse.usage.input_tokens)
print("Tokens en sortie :", reponse.usage.output_tokens)
print("Raison de l'arrêt :", reponse.stop_reason)

Tokens en entrée : 30
Tokens en sortie : 144
Raison de l'arrêt : end_turn


## 2. Les tokens et le coût

Le modèle ne lit pas des mots mais des **tokens**, des morceaux de mots. Tout se compte en tokens : la taille maximale du texte, le prix et la vitesse.

L'API permet de **compter les tokens sans faire d'appel payant** avec `count_tokens`. Comparons la même phrase en français et en wolof.

In [24]:
def compter_tokens(texte):
    resultat = client.messages.count_tokens(
        model=MODELE,
        messages=[{"role": "user", "content": texte}],
    )
    return resultat.input_tokens

phrases = {
    "français": "Bonjour, je voudrais acheter du pain pour ma famille.",
    "wolof":    "Salaam aleekum, dama bëgg jënd mburu ngir sama njaboot.",
    "anglais":  "Hello, I would like to buy some bread for my family.",
}
for langue, phrase in phrases.items():
    print(f"{langue:9} → {compter_tokens(phrase)} tokens")

français  → 32 tokens
wolof     → 39 tokens
anglais   → 25 tokens


💬 **À retenir :** pour un même sens, certaines langues coûtent plus de tokens que d'autres. Le wolof, moins présent dans les données d'entraînement, est souvent découpé en plus petits morceaux.

### Calculer le coût en FCFA
Les prix sont exprimés **en dollars par million de tokens**, avec un prix différent pour l'entrée et la sortie. Les valeurs viennent de votre fichier `.env` (pensez à les vérifier sur la page tarifs).

In [25]:
PRIX_ENTREE = float(os.getenv("PRIX_ENTREE_USD", "3"))    # $ par million de tokens lus
PRIX_SORTIE = float(os.getenv("PRIX_SORTIE_USD", "15"))   # $ par million de tokens écrits
TAUX = float(os.getenv("TAUX_USD_FCFA", "600"))            # FCFA pour 1 $

def cout_fcfa(usage):
    dollars = (usage.input_tokens * PRIX_ENTREE + usage.output_tokens * PRIX_SORTIE) / 1_000_000
    return dollars * TAUX

print(f"Le premier appel a coûté environ {cout_fcfa(reponse.usage):.2f} FCFA")

Le premier appel a coûté environ 1.35 FCFA


✍️ **À vous :** combien coûteraient **1 000** appels comme celui-ci ? Et si votre agent fait 6 appels par demande, pour 200 demandes par jour pendant un mois ?

In [26]:
# Écrivez votre calcul ici

<details>
<summary><b>💡 Voir la solution</b></summary>

```python
cout_un_appel = cout_fcfa(reponse.usage)
print(f"1 000 appels : {cout_un_appel * 1000:.0f} FCFA")
print(f"Un mois d'agent : {cout_un_appel * 6 * 200 * 30:.0f} FCFA")
```
</details>

## 3. Une petite fonction pour simplifier

Pour ne pas réécrire tout l'appel à chaque fois, on crée une fonction `demander`.

In [27]:

def demander(question, systeme="", temperature=None, max_tokens=400):
    options = dict(
        model=MODELE,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": question}],
    )
    if systeme:
        options["system"] = systeme

    # Ce modèle ne prend pas en charge le paramètre temperature.
    rep = client.messages.create(**options)
    return texte_de(rep)


print(demander(
    "Donne un nom pour une boulangerie à Dakar.",
    systeme="Tu réponds avec le nom seul, sans phrase.",
))

Le Fournil de la Teranga


## 4. La température : prévisible ou créatif ?

- **Température basse (0)** : réponses très similaires d'un appel à l'autre. Idéal pour **extraire des données**.
- **Température haute (1)** : réponses plus variées. Utile pour **rédiger**.

In [28]:
question = "Propose un nom pour une boulangerie à Dakar."
consigne = "Réponds avec le nom seul."

print("Température 0 :")
for _ in range(3):
    print("  -", demander(question, consigne, temperature=0))

print("Température 1 :")
for _ in range(3):
    print("  -", demander(question, consigne, temperature=1))

Température 0 :
  - Le Fournil de la Teranga
  - Le Fournil de la Teranga
  - Fournil de la Teranga
Température 1 :
  - Le Pain de Teranga
  - Baguette Teranga
  - Le Pain de Teranga


## 5. Le modèle n'a pas de mémoire

**Point essentiel :** entre deux appels, le modèle a **tout oublié**. Si un chatbot semble se souvenir, c'est parce que le code lui renvoie **tout l'historique** à chaque fois (et on paie cet historique à chaque appel).

In [24]:
print(demander("Je m'appelle Awa et je vends du thiéboudienne."))
print("-" * 60)
print(demander("Comment je m'appelle et qu'est-ce que je vends ?"))

Enchanté, Awa ! Le thiéboudienne, c'est le plat emblématique du Sénégal : le riz au poisson, avec ses légumes (manioc, carotte, chou, aubergine, patate douce) et cette sauce tomate parfumée. 😋

Vous le vendez plutôt à emporter, en livraison, sur un marché, ou dans un petit restaurant ?

Je peux vous aider pour, par exemple :
- **Fixer vos prix** et calculer vos coûts par portion
- **Rédiger une annonce** ou un message pour WhatsApp, Facebook ou Instagram
- **Créer un menu** ou proposer des variantes (thiéboudienne rouge, blanc, au poulet, etc.)
- **Organiser votre production** pour limiter le gaspillage
- **Attirer plus de clients** ou fidéliser ceux que vous avez déjà

Dites-moi ce dont
------------------------------------------------------------
Je n'ai aucun moyen de le savoir : je n'ai pas accès à votre identité ni à des informations sur vous, et rien dans notre conversation ne mentionne votre nom ou votre activité.

Si vous me dites comment vous vous appelez et ce que vous vendez,

Le modèle ne sait pas. Maintenant, on lui **renvoie l'historique** :

In [29]:
historique = [
    {"role": "user", "content": "Je m'appelle Awa et je vends du thiéboudienne."},
    {"role": "assistant", "content": "Enchantée Awa ! Comment puis-je vous aider ?"},
    {"role": "user", "content": "Comment je m'appelle et qu'est-ce que je vends ?"},
]
rep = client.messages.create(model=MODELE, max_tokens=200, messages=historique)
print(texte_de(rep))

Vous vous appelez Awa et vous vendez du thiéboudienne, le célèbre plat sénégalais à base de poisson et de riz. 😊

Si vous voulez, je peux vous aider à rédiger une annonce, trouver des idées pour attirer des clients ou fixer vos prix.


## 6. Chatbot, workflow ou agent ?

C'est **la distinction centrale** du cours :

| | Qui décide des étapes ? | Exemple de prospection |
|---|---|---|
| **Chatbot** | Personne : une question, une réponse | Répond à une question sur la prospection |
| **Workflow** | Le développeur, à l'avance | Extraire → chercher → rédiger, toujours dans cet ordre |
| **Agent** | Le modèle, en cours de route | Décide d'élargir la zone s'il trouve trop peu de résultats |

Un workflow est plus **prévisible et moins cher** ; un agent est plus **souple**. Règle d'or : **ne faites un agent que si vous ne connaissez pas les étapes à l'avance.**

Dans ce notebook, on prépare la première étape de notre agent de prospection : **comprendre la demande**.

## 7. Sorties structurées : obtenir du JSON fiable

Quand un **programme** doit utiliser la réponse, du texte libre ne suffit pas :

In [30]:
print(demander("Quels sont les paramètres de recherche dans : « 5 boulangeries à Dakar Plateau » ?"))

Dans la requête « 5 boulangeries à Dakar Plateau », on distingue trois paramètres :

| Paramètre | Valeur | Rôle |
|---|---|---|
| **Quantité** | 5 | Limite le nombre de résultats attendus |
| **Catégorie / type de lieu** | boulangeries | Définit ce qu'on cherche |
| **Localisation** | Dakar Plateau | Zone géographique : le quartier du Plateau, centre administratif et d'affaires de Dakar (Sénégal) |

**Remarques :**
- La localisation est introduite par la préposition « à », qui indique le lieu de la recherche.
- Aucun critère de tri ou de filtre n'est précisé (note, distance, horaires d'ouverture, prix, etc.). Ces critères restent donc implicites ou laissés au choix du mote


Pour un humain c'est clair, mais votre code ne peut rien en faire. Il nous faut **toujours le même format** :
`{"activite": "boulangerie", "zone": "Dakar Plateau", "nombre": 5}`

### 7.1 Décrire le format avec Pydantic
Pydantic décrit les champs attendus, leur type et leurs limites, puis **vérifie** que les données respectent ces règles.

In [31]:
from pydantic import BaseModel, Field

class Parametres(BaseModel):
    activite: str = Field(description="Type de commerce, au singulier, en français")
    zone: str = Field(description="Quartier ou ville, par exemple Dakar Plateau")
    nombre: int = Field(default=5, ge=1, le=20, description="Nombre de commerces voulus")

# Pydantic sait produire le schéma JSON correspondant :
import json
print(json.dumps(Parametres.model_json_schema(), indent=2, ensure_ascii=False))

{
  "properties": {
    "activite": {
      "description": "Type de commerce, au singulier, en français",
      "title": "Activite",
      "type": "string"
    },
    "zone": {
      "description": "Quartier ou ville, par exemple Dakar Plateau",
      "title": "Zone",
      "type": "string"
    },
    "nombre": {
      "default": 5,
      "description": "Nombre de commerces voulus",
      "maximum": 20,
      "minimum": 1,
      "title": "Nombre",
      "type": "integer"
    }
  },
  "required": [
    "activite",
    "zone"
  ],
  "title": "Parametres",
  "type": "object"
}


### 7.2 Forcer le modèle à remplir ce format
L'astuce : on déclare un **outil** dont le schéma est celui de `Parametres`, et on **oblige** le modèle à l'utiliser avec `tool_choice`. Il est alors contraint de produire exactement nos champs.

Pas besoin de `temperature` ici (ce modèle ne l'accepte plus) : c'est le **schéma imposé** qui rend la sortie prévisible.

In [32]:
def extraire_parametres(phrase):
    outil = {
        "name": "enregistrer_parametres",
        "description": "Enregistre les paramètres de recherche extraits de la demande.",
        "input_schema": Parametres.model_json_schema(),
    }
    rep = client.messages.create(
        model=MODELE_RAPIDE, max_tokens=300,
        system="Tu extrais les paramètres d'une recherche de commerces au Sénégal.",
        tools=[outil],
        tool_choice={"type": "tool", "name": "enregistrer_parametres"},   # obligatoire !
        messages=[{"role": "user", "content": phrase}],
    )
    donnees = next(bloc.input for bloc in rep.content if bloc.type == "tool_use")
    return Parametres.model_validate(donnees)          # validation : erreur si le format est faux

p = extraire_parametres("5 boulangeries à Dakar Plateau")
print(p)
print("Activité :", p.activite, "| Zone :", p.zone, "| Nombre :", p.nombre)

activite='boulangerie' zone='Dakar Plateau' nombre=5
Activité : boulangerie | Zone : Dakar Plateau | Nombre : 5


### 7.3 Que fait Pydantic quand les données sont fausses ?

In [29]:
from pydantic import ValidationError

try:
    Parametres.model_validate({"activite": "boulangerie", "zone": "Dakar", "nombre": 500})
except ValidationError as erreur:
    print("❌ Données refusées :")
    print(erreur)

❌ Données refusées :
1 validation error for Parametres
nombre
  Input should be less than or equal to 20 [type=less_than_equal, input_value=500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal


## 8. ✍️ À vous : les phrases pièges

Faites passer ces cinq phrases dans `extraire_parametres` et observez les résultats. Lesquelles posent problème ?

In [30]:
phrases_pieges = [
    "boulengeries a dakar",                        # fautes d'orthographe
    "Dama bëgg 3 pâtisseries ci Médina",            # français et wolof mélangés
    "des pharmacies aux Almadies",                  # nombre absent
    "une dizaine de quincailleries à Pikine",       # nombre en lettres
    "quelle est la capitale du Sénégal ?",          # hors sujet !
]

# Écrivez une boucle qui affiche le résultat pour chaque phrase

<details>
<summary><b>💡 Voir la solution</b></summary>

```python
for phrase in phrases_pieges:
    print(f"{phrase!r:45} → {extraire_parametres(phrase)}")
```
</details>

🤔 **Qu'avez-vous remarqué pour la phrase hors sujet ?** Le modèle a probablement **inventé** des paramètres, car on l'a obligé à remplir le formulaire.

**Leçon :** un schéma imposé peut pousser le modèle à inventer. La solution : lui donner une **porte de sortie**, un champ `pertinent`.

✍️ **À vous :** ajoutez un champ `pertinent: bool` à une nouvelle classe `Parametres2`, adaptez la consigne système, et testez à nouveau la phrase hors sujet.

In [31]:
# Votre version améliorée ici

<details>
<summary><b>💡 Voir la solution</b></summary>

```python
class Parametres2(BaseModel):
    pertinent: bool = Field(description="false si la demande n'est pas une recherche de commerces")
    activite: str = Field(default="", description="Type de commerce, au singulier")
    zone: str = Field(default="", description="Quartier ou ville")
    nombre: int = Field(default=5, ge=1, le=20)

def extraire_parametres2(phrase):
    outil = {"name": "enregistrer_parametres",
             "description": "Enregistre les paramètres de recherche extraits de la demande.",
             "input_schema": Parametres2.model_json_schema()}
    rep = client.messages.create(
        model=MODELE_RAPIDE, max_tokens=300,
        system=("Tu extrais les paramètres d'une recherche de commerces au Sénégal. "
                "Si la demande n'est pas une recherche de commerces, mets pertinent à false."),
        tools=[outil], tool_choice={"type": "tool", "name": "enregistrer_parametres"},
        messages=[{"role": "user", "content": phrase}])
    return Parametres2.model_validate(next(b.input for b in rep.content if b.type == "tool_use"))

print(extraire_parametres2("quelle est la capitale du Sénégal ?"))
print(extraire_parametres2("5 boulangeries à Dakar Plateau"))
```
</details>

## 🎯 Récapitulatif

| Notion | Ce qu'il faut retenir |
|---|---|
| Token | L'unité que le modèle lit, écrit et facture |
| Coût | (tokens entrée × prix entrée + tokens sortie × prix sortie) ÷ 1 000 000 |
| Température | 0 pour extraire, plus haut pour rédiger |
| Mémoire | Aucune : le code renvoie l'historique à chaque appel |
| Sortie structurée | Un schéma Pydantic + un outil imposé = du JSON fiable |

➡️ **Prochaine étape : notebook 02.** On donne des **outils** au modèle et on construit la **boucle agentique**, le cœur de tout agent.